# Demo | GPU-accelerated Python environment on Windows (Conda + Pixi)

[README.md](README.md) explains how to recreate this environment, and [pyproject.toml](pyproject.toml) defines the dependencies. This notebook validates that the Pixi-managed setup is working on the NVIDIA RTX PRO 6000 GPU.

Why Pixi on Windows: it manages Conda and PyPI dependencies in one project workflow, and lets us pin and reproduce CUDA-capable package combinations (including newer GPU wheels from PyPI when needed).

A similar environment is available on WSL, where dependencies are managed with uv. Keeping both variants aligned makes it easier to reuse existing internal tools across Windows and WSL workflows.

In [4]:
import torch
import platform
import subprocess
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())

try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    print("nvidia-smi exit code:", result.returncode)
    print("nvidia-smi output (first 20 lines):")
    print("\n".join(result.stdout.splitlines()[:20]))
except FileNotFoundError:
    print("nvidia-smi not found in PATH.")



Python executable: c:\Users\wilaca\git\miljodir\op-gis\hosts\6737osl\python\demo-conda-gpu\.pixi\envs\default\python.exe
Python version: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:36:12) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
nvidia-smi exit code: 0
nvidia-smi output (first 20 lines):
Wed Jun 17 20:09:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.97                 Driver Version: 595.97         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+=====================

In [5]:
from importlib import metadata
import importlib.util

packages = [
    ("torch", "torch", True),
    ("torchvision", "torchvision", False),
    ("lightning", "lightning", False),
    ("cupy-cuda13x", "cupy", False),
    ("numpy", "numpy", False),
    ("pandas", "pandas", False),
    ("scikit-learn", "sklearn", False),
    ("xgboost", "xgboost", False),
    ("geopandas", "geopandas", False),
    ("rasterio", "rasterio", False),
    ("shapely", "shapely", False),
    ("pyarrow", "pyarrow", False),
]

def package_specs(dist_name: str, import_name: str):
    spec = importlib.util.find_spec(import_name)
    if spec is None:
        return {
            "status": "not installed",
            "version": "-",
            "module_path": "-",
            "requires_python": "-",
            "summary": "-",
        }

    try:
        md = metadata.metadata(dist_name)
        version = metadata.version(dist_name)
        requires_python = md.get("Requires-Python", "-")
        summary = md.get("Summary", "-")
    except metadata.PackageNotFoundError:
        version = "unknown"
        requires_python = "-"
        summary = "-"

    module_path = getattr(spec, "origin", "-") or "-"
    return {
        "status": "installed",
        "version": version,
        "module_path": module_path,
        "requires_python": requires_python,
        "summary": summary,
    }

print("Package specs in current kernel:\n")
for dist_name, import_name, required in packages:
    info = package_specs(dist_name, import_name)
    tag = "required" if required else "optional"
    print(f"- {dist_name} ({tag})")
    print(f"  version: {info['version']}")
    print(f"  requires_python: {info['requires_python']}")
    print(f"  summary: {info['summary']}\n")

Package specs in current kernel:

- torch (required)
  version: 2.12.1+cu132
  requires_python: >=3.10
  summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration

- torchvision (optional)
  version: 0.2.0
  requires_python: -
  summary: image and video datasets and models for torch deep learning

- lightning (optional)
  version: 2.6.5
  requires_python: >=3.10
  summary: The Deep Learning framework to train, deploy, and ship AI products Lightning fast.

- cupy-cuda13x (optional)
  version: 14.1.1
  requires_python: >=3.10
  summary: CuPy: NumPy & SciPy for GPU

- numpy (optional)
  version: 2.4.6
  requires_python: >=3.11
  summary: Fundamental package for array computing in Python

- pandas (optional)
  version: 3.0.3
  requires_python: >=3.11
  summary: Powerful data structures for data analysis, time series, and statistics

- scikit-learn (optional)
  version: 1.9.0
  requires_python: >=3.11
  summary: A set of python modules for machine learning and data

In [6]:

print("GPU-specific specs (current kernel):")
print("- torch version:", torch.__version__)
print("- CUDA available:", torch.cuda.is_available())
print("- CUDA runtime (torch):", torch.version.cuda)
print("- cuDNN enabled:", torch.backends.cudnn.is_available())
print("- cuDNN version:", torch.backends.cudnn.version())

if torch.cuda.is_available():
    print("- GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        free_mem, total_mem = torch.cuda.mem_get_info(i)
        print(f"\nGPU {i}: {props.name}")
        print(f"  compute capability: {props.major}.{props.minor}")
        print(f"  total memory (GiB): {props.total_memory / (1024**3):.2f}")
        print(f"  memory free/total (GiB): {free_mem / (1024**3):.2f} / {total_mem / (1024**3):.2f}")
        print(f"  multiprocessor count: {props.multi_processor_count}")
        print(f"  max threads per block: {props.max_threads_per_block}")
else:
    print("CUDA is not available. Check driver and environment packages.")

print("\nCuPy runtime check:")
try:
    import cupy as cp

    print("- cupy version:", cp.__version__)
    runtime = cp.cuda.runtime.runtimeGetVersion()
    drv = cp.cuda.runtime.driverGetVersion()
    print("- CUDA runtime version (cupy raw):", runtime)
    print("- CUDA driver version (cupy raw):", drv)

    dev_count = cp.cuda.runtime.getDeviceCount()
    print("- CuPy detected GPU count:", dev_count)
    for i in range(dev_count):
        props = cp.cuda.runtime.getDeviceProperties(i)
        name = props["name"].decode() if isinstance(props["name"], (bytes, bytearray)) else props["name"]
        major = props.get("major", "?")
        minor = props.get("minor", "?")
        total_mem = props.get("totalGlobalMem", 0)
        print(f"  GPU {i}: {name}")
        print(f"    compute capability: {major}.{minor}")
        print(f"    total memory (GiB): {total_mem / (1024**3):.2f}")
except Exception as e:
    print("- CuPy not available or failed to query runtime:", repr(e))

GPU-specific specs (current kernel):
- torch version: 2.12.1+cu132
- CUDA available: True
- CUDA runtime (torch): 13.2
- cuDNN enabled: True
- cuDNN version: 92000
- GPU count: 1

GPU 0: NVIDIA RTX PRO 6000 Blackwell Workstation Edition
  compute capability: 12.0
  total memory (GiB): 95.59
  memory free/total (GiB): 93.12 / 95.59
  multiprocessor count: 188
  max threads per block: 1024

CuPy runtime check:
- cupy version: 14.1.1
- CUDA runtime version (cupy raw): 13020
- CUDA driver version (cupy raw): 13020
- CuPy detected GPU count: 1
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Workstation Edition
    compute capability: 12.0
    total memory (GiB): 95.59
